In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Jupyter has no __file__ — use script dir when running .py, else notebook working directory
def _project_root() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()

_env = _project_root() / ".env"
print(f".env loaded from {_env}: {load_dotenv(_env)}")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

# Chat models: use ChatOpenAI (not the legacy OpenAI completions LLM)
model = ChatOpenAI(model="gpt-4.1-mini")
response = model.invoke("The sky is")
print(response.content)


c:\Users\1036506\OneDrive - Blue Yonder\Desktop\learning_langchain\ch01\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


.env loaded from c:\Users\1036506\OneDrive - Blue Yonder\Desktop\learning_langchain\ch01\.env: True
OPENAI_API_KEY set: True
The sky is vast and often appears blue during the day due to the scattering of sunlight by the Earth's atmosphere. At sunrise and sunset, it can display a range of beautiful colors like red, orange, pink, and purple. At night, the sky becomes dark and is dotted with stars, planets, and sometimes the moon. Is there something specific you'd like to know about the sky?


In [2]:
#OpenAI differentiate messages sent to and from the model into user, assistant, and system roles (here role denotes the type of content the message contains):
#System role Used for instructions the model should use to answer a user question
#User role - Used for the user’s query and any other content produced by the user
#Assistant role -Used for content generated by the model

from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4.1-mini")
prompt = [HumanMessage("What is the capital of France?")]
response = model.invoke(prompt)
print(response.content)
print(response)



The capital of France is Paris.
content='The capital of France is Paris.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c5385b4086', 'id': 'chatcmpl-DNYkMQMjc0ABjoCJeE8IOSpUqrfdN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d28e3-5b46-7661-941e-740646a6d4c5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [3]:
#human and system messages
from langchain_core.messages import HumanMessage, SystemMessage

system_msg = SystemMessage(
    """You are a helpful assistant that responds to questions with three
exclamation marks."""
)
human_msg = HumanMessage("What is the capital of France?")
response_messages = model.invoke([system_msg, human_msg])
print(response_messages.content)

The capital of France is Paris!!!


In [4]:
#LangChain provides prompt template interfaces that make it easy to 
# construct prompts with dynamic inputs:
from langchain_core.prompts import PromptTemplate
template = PromptTemplate.from_template("""Answer the question based on the
context below. If the question cannot be answered using the information
provided, answer with "I don't know".
Context: {context}
Question: {question}
Answer: """)
template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})

StringPromptValue(text='Answer the question based on the\ncontext below. If the question cannot be answered using the information\nprovided, answer with "I don\'t know".\nContext: The most recent advancements in NLP are being driven by Large\nLanguage Models (LLMs). These models outperform their smaller\ncounterparts and have become invaluable for developers who are creating\napplications with NLP capabilities. Developers can tap into these\nmodels through Hugging Face\'s `transformers` library, or by utilizing\nOpenAI and Cohere\'s offerings through the `openai` and `cohere`\nlibraries, respectively.\nQuestion: Which model providers offer LLMs?\nAnswer: ')

In [5]:
# both `template` and `model` can be reused many times
template = PromptTemplate.from_template("""Answer the question based on the
context below. If the question cannot be answered using the information
provided, answer with "I don't know".
Context: {context}
Question: {question}
Answer: """)
model = ChatOpenAI(model="gpt-4.1-mini")
# `prompt` and `completion` are the results of using template and model once
prompt = template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})
completion = model.invoke(prompt)
print(completion.content)

OpenAI and Cohere offer LLMs.


If you’re looking to build an AI chat application, the ChatPromptTemplate can be
used instead to provide dynamic inputs based on the role of the chat message:

In [6]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
('system', '''Answer the question based on the context below. If the
question cannot be answered using the information provided, answer with
"I don\'t know".'''),
('human', 'Context: {context}'),
('human', 'Question: {question}'),
])
template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})

ChatPromptValue(messages=[SystemMessage(content='Answer the question based on the context below. If the\nquestion cannot be answered using the information provided, answer with\n"I don\'t know".', additional_kwargs={}, response_metadata={}), HumanMessage(content="Context: The most recent advancements in NLP are being driven by Large\nLanguage Models (LLMs). These models outperform their smaller\ncounterparts and have become invaluable for developers who are creating\napplications with NLP capabilities. Developers can tap into these\nmodels through Hugging Face's `transformers` library, or by utilizing\nOpenAI and Cohere's offerings through the `openai` and `cohere`\nlibraries, respectively.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Question: Which model providers offer LLMs?', additional_kwargs={}, response_metadata={})])

In [7]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
# both `template` and `model` can be reused many times
template = ChatPromptTemplate.from_messages([
('system', '''Answer the question based on the context below. If the
question cannot be answered using the information provided, answer
with "I don\'t know".'''),
('human', 'Context: {context}'),
('human', 'Question: {question}'),
])
model = ChatOpenAI(model="gpt-4.1-mini")
# `prompt` and `completion` are the results of using template and model once
prompt = template.invoke({
"context": """The most recent advancements in NLP are being driven by
Large Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})

response = model.invoke(prompt)
print(response.content)
print(response)

The model providers that offer LLMs mentioned are Hugging Face, OpenAI, and Cohere.
content='The model providers that offer LLMs mentioned are Hugging Face, OpenAI, and Cohere.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 140, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c5385b4086', 'id': 'chatcmpl-DNYkQuaEBXwmGsuCUJGQUVqDFInd1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d28e3-6afb-7111-9f6b-f52189a37f3f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 140, 'output_tokens': 21, 'total_tokens': 161, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

## Specific format outputs of LLMs
<span style="color: crimson; font-size: 20px;">**JSON Output**</span>


In [8]:
from pydantic import BaseModel, Field

class AnswerWithJustification(BaseModel):
    """An answer to the user's question along with justification for the answer."""

    answer: str = Field(description="The answer to the user's question")
    justification: str = Field(description="Justification for the answer")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
structured_llm = llm.with_structured_output(AnswerWithJustification)
result = structured_llm.invoke(
    "What weighs more, a pound of bricks or a pound of feathers"
)
print(result)

answer='They both weigh the same.' justification='A pound is a unit of weight, so a pound of bricks and a pound of feathers both weigh exactly one pound, regardless of their different volumes or densities.'


## Other Machine-Readable Formats with Output Parsers
<span style="color: crimson; font-size: 20px;">**Output Parser**</span>


In [9]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
items = parser.invoke("apple, banana, cherry")
print(items)

['apple', 'banana', 'cherry']


## Assembling the Many Pieces of an LLM Application

In [10]:
model = ChatOpenAI(model="gpt-4.1-mini")
completion = model.invoke('Hi there!')
# Hi!
completions = model.batch(['Hi there!', 'Bye!'])
# ['Hi!', 'See you!']
for token in model.stream('Bye!'):
    print(token)
# Good
# bye
# ! 

content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019d28e3-81cf-7881-8281-e88bea426be0' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='Good' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019d28e3-81cf-7881-8281-e88bea426be0' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='bye' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019d28e3-81cf-7881-8281-e88bea426be0' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='!' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019d28e3-81cf-7881-8281-e88bea426be0' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' Have' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019d28e3-81cf-7881-8281-e88bea426be0' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' a' additional_kwargs={} response_metadata={'model_pr